# AICTE | IBM SkillsBuild — Data Analytics with AI
## Combined Pipeline: Masterclass 1–4

This notebook runs the full project end to end, in four parts:

1. **Masterclass 1** — Raw data → clean, analysis-ready dataset
2. **Masterclass 2** — Exploratory Data Analysis (visualizations, observations, insights, hypotheses, recommendations)
3. **Masterclass 3** — RFM features, leakage-safe churn label, logistic regression prediction model
4. **Masterclass 4** — Supermarket Sales Analysis (separate dataset)

**Inputs needed** (upload both before running):
- `AI_-_Data-_Make_Data_Intelligent_-_Masterclass_1_-_Practice_Dataset.xlsx` (used by Parts 1–3)
- `SUPER_MARKET_DATA.xlsx` (used by Part 4)


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

---
# PART 1 — Masterclass 1: Raw Data → Clean, Analysis-Ready Data
Deliverables: Clean Dataset, Data Dictionary, Business Questions.

In [ ]:
FILENAME_MC1 = "AI_-_Data-_Make_Data_Intelligent_-_Masterclass_1_-_Practice_Dataset.xlsx"  # <-- replace if your uploaded filename differs

df_raw = pd.read_excel(FILENAME_MC1)
print("Raw shape:", df_raw.shape)
df_raw.head()

In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())
print("\nMissing values:\n", df_raw.isna().sum())
print("\nRegion variants:", sorted(df_raw['Region'].dropna().unique()))
print("\nCategory variants:", sorted(df_raw['Category'].dropna().unique()))
print("\nNegative Quantity rows:", (df_raw['Quantity'] < 0).sum())
print("\nSample messy Revenue values:", df_raw['Revenue'].unique()[:10])

In [ ]:
df = df_raw.copy()

# 1. Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")

# 2. Fix Order_Date: mix of real dates and 'DD-MM-YYYY' text, some impossible (e.g. 31-02-2026)
def parse_date(x):
    if isinstance(x, pd.Timestamp):
        return x
    try:
        return pd.to_datetime(x, format='%d-%m-%Y')
    except Exception:
        return pd.NaT

df['Order_Date'] = df['Order_Date'].apply(parse_date)
print("Unreadable/impossible dates dropped:", df['Order_Date'].isna().sum())
df = df.dropna(subset=['Order_Date'])

# 3. Standardize inconsistent category text (Delhi / delhi / DELHI -> Delhi)
df['Region'] = df['Region'].str.strip().str.title()
df['Category'] = df['Category'].str.strip().str.title()

# 4. Clean Revenue: strip currency symbols/text, keep numeric value only
def clean_revenue(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r'[^0-9.]', '', str(x))
    return float(s) if s else np.nan
df['Revenue'] = df['Revenue'].apply(clean_revenue)

# 5. Fix Quantity: negative -> positive, 0 treated as missing, fill with median
df['Quantity'] = df['Quantity'].abs()
df.loc[df['Quantity'] == 0, 'Quantity'] = np.nan
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# 6. Handle remaining missing values
df['Customer_ID'] = df['Customer_ID'].fillna('UNKNOWN')
df['Product'] = df['Product'].fillna('Unknown Product')
df['Region'] = df['Region'].fillna('Unknown')
df = df.dropna(subset=['Revenue'])
df['Profit'] = df['Profit'].fillna(df.groupby('Category')['Profit'].transform('median'))

print("\nCleaned shape:", df.shape)
print(df.isna().sum())

In [ ]:
# Analytical (enrichment) columns
df['Order_Month'] = df['Order_Date'].dt.strftime('%b')
df['Order_Year'] = df['Order_Date'].dt.year

q1, q2 = df['Revenue'].quantile([0.33, 0.66])
df['Order_Value_Category'] = df['Revenue'].apply(lambda v: 'Low' if v <= q1 else ('Medium' if v <= q2 else 'High'))

order_counts = df['Customer_ID'].value_counts()
df['Customer_Segment'] = df['Customer_ID'].map(lambda c: 'Repeat' if order_counts[c] > 1 else 'New')

threshold = df['Revenue'].quantile(0.90)
df['Revenue_Category'] = df['Revenue'].apply(lambda v: 'High Impact' if v >= threshold else 'Standard')

zone_map = {'Delhi':'North','Bangalore':'South','Chennai':'South','Hyderabad':'South',
            'Mumbai':'West','Pune':'West','Kolkata':'East','Unknown':'Unknown'}
df['Region_Zone'] = df['Region'].map(zone_map)

df.head()

In [ ]:
data_dictionary = pd.DataFrame([
    ["Order_ID", "Identifier", "Uniquely labels each transaction record"],
    ["Order_Date", "Date/Time", "Date the order was placed (cleaned to valid calendar dates)"],
    ["Customer_ID", "Identifier", "Unique customer reference; 'UNKNOWN' where missing in raw data"],
    ["Product", "Categorical", "Product name purchased"],
    ["Category", "Categorical", "Product category (standardized casing)"],
    ["Region", "Categorical", "City where the order was placed (standardized casing)"],
    ["Quantity", "Numerical", "Units purchased (negative values corrected; missing filled with median)"],
    ["Revenue", "Numerical", "Sales value in INR (currency symbols/text stripped)"],
    ["Profit", "Numerical", "Profit in INR (missing filled with category median)"],
    ["Order_Month", "Derived", "Month name extracted from Order_Date"],
    ["Order_Year", "Derived", "Year extracted from Order_Date"],
    ["Order_Value_Category", "Derived", "Revenue bucketed into Low / Medium / High"],
    ["Customer_Segment", "Derived", "New (1 order) vs Repeat (2+ orders)"],
    ["Revenue_Category", "Derived", "Top 10% of transactions flagged 'High Impact'"],
    ["Region_Zone", "Derived", "City rolled up into sales territory"],
], columns=["Column", "Type", "Description"])
data_dictionary

In [ ]:
business_questions = pd.DataFrame([
    ["Sales", "Which months generate the most revenue?", "Guides inventory and staffing planning around peak months."],
    ["Product", "Which product categories underperform and might need review?", "Flags categories for a discount, bundling, or delisting decision."],
    ["Customer", "Which customers may be at risk (low frequency, low recent spend)?", "Feeds directly into a retention/win-back campaign decision."],
    ["Customer", "Who are the highest-value (top revenue) customers?", "Identifies who to prioritize for loyalty or account management."],
    ["Region", "Which regions/zones are underperforming relative to others?", "Supports a decision on where to focus marketing or open new stores."],
], columns=["Category", "Business Question", "Why it matters"])
business_questions

In [ ]:
df.to_csv('cleaned_masterclass1_dataset.csv', index=False)
print("Saved: cleaned_masterclass1_dataset.csv  (used as input to Parts 2 and 3)")

---
# PART 2 — Masterclass 2: Exploratory Data Analysis
Deliverables: 5 Visualizations, 5 Observations, 5 Insights, 3 Hypotheses, 3 Recommendations.

In [ ]:
# Visualization 1 - Monthly Revenue
df['Month'] = df['Order_Date'].dt.to_period('M')
monthly_revenue = df.groupby('Month')['Revenue'].sum().sort_index()

plt.figure(figsize=(10,5))
monthly_revenue.plot(kind='line', marker='o', color='#4C72B0')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month'); plt.ylabel('Revenue')
plt.tight_layout(); plt.show()

print("Highest revenue month:", monthly_revenue.idxmax(), "-> ₹{:,.0f}".format(monthly_revenue.max()))
print("Lowest revenue month:", monthly_revenue.idxmin(), "-> ₹{:,.0f}".format(monthly_revenue.min()))

In [ ]:
# Visualization 2 - Category Revenue
category_revenue = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)

plt.figure(figsize=(8,5))
category_revenue.plot(kind='bar', color='#55A868')
plt.title('Total Revenue by Category')
plt.xlabel('Category'); plt.ylabel('Revenue'); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

print(category_revenue)

In [ ]:
# Visualization 3 - Customer Segments
unique_customers = df.drop_duplicates('Customer_ID')
segment_counts = unique_customers['Customer_Segment'].value_counts()
segment_pct = (segment_counts / segment_counts.sum() * 100).round(2)

plt.figure(figsize=(6,5))
bars = plt.bar(segment_counts.index, segment_counts.values, color=['#4C72B0','#DD8452'])
plt.title('Customer Segment Distribution')
plt.xlabel('Customer Segment'); plt.ylabel('Number of Unique Customers')
for bar, pct in zip(bars, segment_pct.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{pct}%", ha='center', va='bottom')
plt.tight_layout(); plt.show()

print(segment_counts); print(segment_pct)

In [ ]:
# Visualization 4 - Regional Revenue
region_revenue = df.groupby('Region')['Revenue'].sum().sort_values(ascending=False)

plt.figure(figsize=(8,5))
region_revenue.sort_values().plot(kind='barh', color='#C44E52')
plt.title('Total Revenue by Region')
plt.xlabel('Revenue'); plt.ylabel('Region')
plt.tight_layout(); plt.show()

print(region_revenue)

In [ ]:
# Visualization 5 - Top Products by Profit
product_profit = df.groupby('Product')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(8,5))
product_profit.sort_values().plot(kind='barh', color='#8172B2')
plt.title('Top 10 Products by Profit')
plt.xlabel('Profit'); plt.ylabel('Product')
plt.tight_layout(); plt.show()

print(product_profit)

**Observations, Insights, Hypotheses, Recommendations**

**Observations**
1. Monthly Revenue peaked in one month and hit its lowest in another (see Viz 1 output above).
2. Electronics led category revenue; Beauty trailed (see Viz 2).
3. Repeat customers made up the large majority of the unique customer base (see Viz 3).
4. Regional revenue stayed fairly close across metro regions (see Viz 4).
5. A named product led profit, but unlabeled product entries also captured meaningful profit (see Viz 5).

**Insights**
1. **Extreme Revenue Seasonality** — the gap between peak and lowest revenue months signals cash-flow instability across the year.
2. **Core Category Dependency** — the business is heavily dependent on Electronics, which outpaces all other categories combined.
3. **Customer Retention Strength vs. Acquisition Risk** — revenue relies on a loyal repeat-buyer base, but low new-customer acquisition limits growth.
4. **Even Metropolitan Reach** — regional revenue is fairly balanced, suggesting broad demand rather than concentration in one city.
5. **Data Quality Risk** — profit tied to blank/unlabeled products masks true product-level performance.

**Hypotheses** (possible explanations — not confirmed facts)
1. **Seasonal Demand or Promotions** — the peak month might reflect seasonal campaigns; the low month could be a post-seasonal lull.
2. **Stockout or Inventory Shortages** — the low month may reflect supply-chain disruptions in top categories.
3. **Pricing or Catalog Changes** — the peak could reflect bulk/high-ticket orders; the low month might stem from delistings or discount changes.

**Recommendations**
1. **Build peak-period cash reserves** to cover operating costs during low-revenue months.
2. **Diversify off-peak demand** with targeted campaigns, bundles, or loyalty incentives.
3. **Audit the product catalog** to eliminate blank SKU entries and maintain safety stock for top categories.


---
# PART 3 — Masterclass 3: RFM Features & Churn Prediction
Deliverables: RFM features, leakage-safe churn label, trained logistic regression model, evaluation metrics, customer risk table.

In [ ]:
# Leakage-safe historical split: features from an earlier window, churn label from a later window
CUTOFF = pd.Timestamp('2026-06-01')

hist = df[df['Order_Date'] < CUTOFF]
future = df[df['Order_Date'] >= CUTOFF]

print("Historical period:", hist['Order_Date'].min(), "-", hist['Order_Date'].max(), "| rows:", len(hist))
print("Future period    :", future['Order_Date'].min(), "-", future['Order_Date'].max(), "| rows:", len(future))

In [ ]:
customer_df = hist.groupby('Customer_ID').agg(
    Order_Count=('Order_ID', 'count'),
    Total_Revenue=('Revenue', 'sum'),
    First_Purchase=('Order_Date', 'min'),
    Last_Purchase=('Order_Date', 'max')
).reset_index()
customer_df['Avg_Order_Value'] = customer_df['Total_Revenue'] / customer_df['Order_Count']

# RFM
customer_df['Recency']   = (CUTOFF - customer_df['Last_Purchase']).dt.days
customer_df['Frequency'] = customer_df['Order_Count']
customer_df['Monetary']  = customer_df['Total_Revenue']

# Churn label from the FUTURE period (not from the RFM features themselves - avoids leakage)
future_customers = set(future['Customer_ID'].unique())
customer_df['Churn_Status'] = customer_df['Customer_ID'].apply(lambda c: 0 if c in future_customers else 1)

print("Customers:", customer_df.shape[0])
print(customer_df['Churn_Status'].value_counts())
customer_df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

features = ['Recency', 'Frequency', 'Monetary', 'Avg_Order_Value']
X = customer_df[features]
y = customer_df['Churn_Status']

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, customer_df['Customer_ID'], test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall   : {rec*100:.2f}%")
print("Confusion Matrix:\n", cm)

In [ ]:
X_all_scaled = scaler.transform(customer_df[features])
customer_df['Churn_Probability'] = model.predict_proba(X_all_scaled)[:, 1]

def risk_level(p):
    if p < 0.40: return 'Low Risk'
    elif p < 0.70: return 'Medium Risk'
    else: return 'High Risk'

customer_df['Risk_Level'] = customer_df['Churn_Probability'].apply(risk_level)
risk_table = customer_df[['Customer_ID','Recency','Frequency','Monetary','Churn_Probability','Risk_Level']]
risk_table = risk_table.sort_values('Churn_Probability', ascending=False)

print("Top 20 highest-risk customers:")
print(risk_table.head(20).to_string(index=False))

risk_table.to_csv('customer_risk_table.csv', index=False)
customer_df.to_csv('customer_level_dataset.csv', index=False)

---
# PART 4 — Masterclass 4: Supermarket Sales Analysis
Separate dataset (500 transactions). Deliverables: highest-selling product, best branch, top category, top payment method, member-vs-normal spend, average rating.

In [ ]:
FILENAME_MC4 = "SUPER_MARKET_DATA.xlsx"  # <-- replace if your uploaded filename differs
sm_df = pd.read_excel(FILENAME_MC4)
print(sm_df.shape)
sm_df.head()

In [ ]:
print(sm_df.isna().sum())
sm_df['Sales_check'] = sm_df['Quantity'] * sm_df['Unit Price']
print("Max diff between provided Sales and Quantity*Unit Price:", (sm_df['Sales_check']-sm_df['Sales']).abs().max())

In [ ]:
top_products = sm_df.groupby('Product')['Sales'].sum().sort_values(ascending=False)
branch_sales = sm_df.groupby(['Branch','City'])['Sales'].sum().sort_values(ascending=False)
category_sales = sm_df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
payment_counts = sm_df['Payment'].value_counts()
avg_spend = sm_df.groupby('Customer Type')['Sales'].mean()
avg_rating = sm_df['Rating'].mean()

print("Top product:", top_products.index[0], "-> ₹{:,.2f}".format(top_products.iloc[0]))
print("Best branch:", branch_sales.index[0])
print("Top category:", category_sales.index[0], "-> ₹{:,.2f}".format(category_sales.iloc[0]))
print("Top payment method:", payment_counts.index[0], f"({payment_counts.iloc[0]} transactions)")
print("Avg spend by customer type:\n", avg_spend)
print(f"Average rating: {avg_rating:.2f} / 5")

In [ ]:
top_products.head(10).sort_values().plot(kind='barh', figsize=(8,5), color='#4C72B0', title='Top 10 Products by Sales')
plt.xlabel('Sales (₹)'); plt.tight_layout(); plt.show()

In [ ]:
labels = [f"{b} ({c})" for b, c in branch_sales.index]
plt.figure(figsize=(6,5))
plt.bar(labels, branch_sales.values, color='#DD8452')
plt.title('Total Sales by Branch'); plt.ylabel('Sales (₹)')
plt.tight_layout(); plt.show()

In [ ]:
category_sales.sort_values().plot(kind='barh', figsize=(8,5), color='#55A868', title='Total Sales by Category')
plt.xlabel('Sales (₹)'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6,6))
plt.pie(payment_counts.values, labels=payment_counts.index, autopct='%1.0f%%', startangle=90,
        colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.title('Payment Method Distribution'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.bar(avg_spend.index, avg_spend.values, color=['#4C72B0','#DD8452'])
plt.title('Average Transaction Value: Member vs Normal'); plt.ylabel('Average Sales (₹)')
for i, v in enumerate(avg_spend.values):
    plt.text(i, v+3, f"₹{v:.2f}", ha='center')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(sm_df['Rating'], bins=15, color='#8172B2', edgecolor='white')
plt.axvline(avg_rating, color='red', linestyle='--', label=f"Mean = {avg_rating:.2f}")
plt.title('Customer Rating Distribution'); plt.xlabel('Rating'); plt.ylabel('Number of Transactions')
plt.legend(); plt.tight_layout(); plt.show()

**Business Decisions**
- Keep more stock of high-selling products and categories.
- Study why the best-performing branch outperforms the others and apply learnings elsewhere.
- Continue supporting the most-used payment method while keeping other options available.
- Use the average rating as a baseline to track service-quality improvements.
- Since Member and Normal customers spend similarly per visit, design membership offers around visit frequency and retention rather than assuming higher per-visit spend.


---
# End of Pipeline
All four masterclasses are complete: cleaned dataset → EDA → churn prediction model → supermarket sales analysis.
Outputs saved: `cleaned_masterclass1_dataset.csv`, `customer_level_dataset.csv`, `customer_risk_table.csv`.
